**TickNet-L giữa kỳ: Label smoothing / LR**

Bật GPU/Internet, attach midd-32 và midd-224. Notebook pin commit `a313bb79a743e29113b3fe19cf62496e061be9da` trên nhánh `experiment/lv1-midterm-smoothing-lr`. Chạy baseline và ứng viên với cùng validation; không tự đánh giá test. Đổi EPOCHS/STOP_AFTER có chủ đích trước Run All. Xem docs/experiments/BRANCH_GUIDE.md để chạy các đối chứng khác.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys, time, shutil
BRANCH = 'experiment/lv1-midterm-smoothing-lr'
REVISION = 'a313bb79a743e29113b3fe19cf62496e061be9da'
REPO = Path('/kaggle/working/l-midterm-smoothing-lr')
DATA = Path('/kaggle/working/mid_experiment_data')
RUNS = Path('/kaggle/working/l-midterm-smoothing-lr_runs')
VARIANTS = ['Mid32', 'Mid224']
CONFIGS = ['configs/midterm/baseline.json', 'configs/midterm/experiment.json']
EPOCHS = 200
STOP_AFTER = None  # 100 để chạy phiên đầu của lịch tổng 200 epoch
NUM_WORKERS = 2  # baseline và ứng viên đều dùng cùng giá trị
SEED = 42
RESUME = False  # khôi phục nguyên RUNS từ ZIP, giữ recipe và bật True ở phiên sau

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--no-checkout', '--branch', BRANCH, 'https://github.com/khuonglaptri3/TickNets.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', REVISION], check=True)
subprocess.run(['git', '-C', str(REPO), 'sparse-checkout', 'set', '--no-cone', '/*', '!/data/', '!/checkpoints/', '!/docs/results/'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REVISION], check=True)
actual = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert actual == REVISION, (actual, REVISION)
import torch, torchvision
assert torch.cuda.is_available(), 'Bật GPU trước khi train 200 epoch'
print('Source:',actual,'torch:',torch.__version__,'GPU:',torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'],check=True)


In [ ]:
from PIL import Image
CLASSES = ('bird','cat','dog','frog','horse')
INPUT = Path('/kaggle/input')
EXTENSIONS = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff','.ppm','.pgm'}
def locate(variant):
    candidates = []
    for train in INPUT.rglob('train'):
        root = train.parent
        if root.name.lower() != variant.lower():
            continue
        if all((root/split/name).is_dir() for split in ('train','test') for name in CLASSES):
            candidates.append(root)
    if len(candidates)!=1:
        raise ValueError(f'Expected one {variant}; found {candidates}')
    return candidates[0]

DATA.mkdir(exist_ok=True)
rows, memberships = [], {}
for variant in VARIANTS:
    source = locate(variant)
    size = 32 if variant == 'Mid32' else 224
    keys = {}
    for split, expected in [('train',5000),('test',50)]:
        keys[split] = []
        for name in CLASSES:
            images = sorted(p for p in (source/split/name).iterdir() if p.suffix.lower() in EXTENSIONS)
            assert len(images)==expected, (variant,split,name,len(images))
            with Image.open(images[0]) as image:
                assert image.size==(size,size), (variant,image.size)
            for image in images:
                key = name+'/'+image.name
                keys[split].append(key)
                rows.append(dict(variant=variant,split=split,class_name=name,filename=image.name))
    assert not set(keys['train']).intersection(keys['test'])
    memberships[variant] = keys
    link = DATA/variant
    if link.exists() or link.is_symlink():
        assert link.resolve()==source.resolve(), 'Dataset link differs; restart clean session'
    else:
        link.symlink_to(source,target_is_directory=True)
if len(VARIANTS)==2:
    assert memberships['Mid32']==memberships['Mid224'], 'Paired filenames do not match'
with (DATA/'split_manifest.csv').open('w',newline='') as handle:
    writer=csv.DictWriter(handle,fieldnames=['variant','split','class_name','filename'])
    writer.writeheader();writer.writerows(rows)
print('Data:',DATA,'manifest SHA256:',hashlib.sha256((DATA/'split_manifest.csv').read_bytes()).hexdigest())
RUNS.mkdir(exist_ok=True)
archived_manifest = RUNS/'source_split_manifest.csv'
if RESUME:
    assert archived_manifest.is_file(), 'Restore original RUNS artifact before resume'
    assert archived_manifest.read_bytes()==(DATA/'split_manifest.csv').read_bytes(), 'Source manifest changed'
else:
    archived_manifest.write_bytes((DATA/'split_manifest.csv').read_bytes())
(RUNS/'source_revision.txt').write_text(REVISION+'\n')


In [ ]:
RUNS.mkdir(exist_ok=True)
for variant in VARIANTS:
    for config in CONFIGS:
        label = Path(config).stem
        output = RUNS/f'{variant.lower()}_{label}_seed{SEED}'
        command=[sys.executable,'train_mid_experiment.py','--config',config,'--variant',variant,'--data-root',str(DATA),'--output-dir',str(output),'--epochs',str(EPOCHS),'--seed',str(SEED),'--num-workers',str(NUM_WORKERS),'--device','cuda:0']
        if STOP_AFTER is not None:
            command += ['--stop-after-epoch',str(STOP_AFTER)]
        if RESUME:
            command += ['--resume',str(output/'last.pt')]
        start=time.monotonic()
        subprocess.run(command,cwd=REPO,check=True)
        print(variant,label,'seconds:',round(time.monotonic()-start,1))


In [ ]:
summary=[]
for run in sorted(RUNS.iterdir()):
    if not (run/'epochs.csv').is_file():
        continue
    config=json.loads((run/'config.json').read_text())
    with (run/'epochs.csv').open() as handle:
        epochs=list(csv.DictReader(handle))
    valid=[row for row in epochs if row['val_top1']]
    best=max(valid,key=lambda row:(float(row['val_top1']),-float(row['val_loss']))) if valid else epochs[-1]
    summary.append(dict(run=run.name,variant=config['variant'],seed=config['seed'],revision=config['git_revision'],model=config['model'],best_epoch=int(best['epoch']),val_top1=best['val_top1'],val_loss=best['val_loss'],parameters=config['complexity']['learnable_parameters'],gflops=config['complexity']['flops']/1e9,validation_sha256=config['validation_sha256']))
if summary:
    with (RUNS/'validation_summary.csv').open('w',newline='') as handle:
        writer=csv.DictWriter(handle,fieldnames=list(summary[0]));writer.writeheader();writer.writerows(summary)
    for row in summary: print(row)
archive=shutil.make_archive(str(RUNS)+'_artifacts','zip',root_dir=RUNS.parent,base_dir=RUNS.name)
print('Download artifact:',archive)
print('Test chỉ chạy riêng sau khi chọn cấu hình bằng validation; xem MIDTERM_PROTOCOL.md')


**Đánh giá cuối**

Sau khi chốt cấu hình, dùng lệnh `python train_mid_experiment.py --data-root <DATA> --variant Mid32 --evaluate <RUN>/best_val.pt --output-dir <fresh output>` trong thư mục REPO. Không dùng test để chọn recipe. Khi retrain đủ 25.000 ảnh, chọn trước số epoch và dùng `--full-train`; cuối cùng đánh giá `last.pt`. Đây là các bước riêng, notebook không tự chạy test.